In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv("cardekho_dataset.csv")
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## EDA

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB


In [4]:
df.drop("Unnamed: 0",axis=1,inplace=True)

In [5]:
# df=df.drop_duplicates()

In [6]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [7]:
df.drop(["car_name","brand"],axis=1,inplace=True)

In [8]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [9]:
df["transmission_type"].unique()

array(['Manual', 'Automatic'], dtype=object)

In [10]:
cat_features=[f for f in df.columns if df[f].dtype=='O']
num_features=[f for f in df.columns if df[f].dtype!="O"]
discrete_features=[f for f in num_features if len(df[f].unique())<=25]
continous_features=[f for f in num_features if f not in discrete_features]

In [11]:
cat_features,num_features,discrete_features,continous_features

(['model', 'seller_type', 'fuel_type', 'transmission_type'],
 ['vehicle_age',
  'km_driven',
  'mileage',
  'engine',
  'max_power',
  'seats',
  'selling_price'],
 ['vehicle_age', 'seats'],
 ['km_driven', 'mileage', 'engine', 'max_power', 'selling_price'])

## encoding and train test split

In [12]:
X=df.drop("selling_price",axis=1)
y=df["selling_price"]

In [13]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X["model"] = le.fit_transform(X["model"])

In [14]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=3)

from sklearn.preprocessing import OneHotEncoder,StandardScaler
ohe=OneHotEncoder(drop="first")
ss=StandardScaler()

col=X.select_dtypes(exclude="object").columns

from sklearn.compose import ColumnTransformer
preprocessor=ColumnTransformer(
    [
        ("OneHotEncoder",ohe,['seller_type', 'fuel_type', 'transmission_type']),
        ("StandardScaler",ss,col)
    ],remainder="passthrough"
)

X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

In [15]:
X_train.shape

(12328, 14)

## model

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

In [17]:
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

In [18]:
def model_performance(tru,pred):
    mae=mean_absolute_error(tru,pred)
    mse=mean_squared_error(tru,pred)
    rmse=np.sqrt(mse)
    score=r2_score(tru,pred)
    print("mean squared error is",mse)
    print("root mean squared error is",rmse)
    print(" r2 score",score)
    print("----------------")

In [19]:
models={
    "LinearRegression":LinearRegression(),
    "Ridge":Ridge(),
    "Lasso":Lasso(),
    "DecisionTreeRegressor":DecisionTreeRegressor(),
    "RandomForestRegressor":RandomForestRegressor(),
    "KNeighborsRegressor":KNeighborsRegressor(),
    "SVR":SVR()
}

for name,model in models.items():
    model.fit(X_train,y_train)
    y_pred=model.predict(X_test)

    print("using",name)
    model_performance(y_test,y_pred)

using LinearRegression
mean squared error is 218204361545.2001
root mean squared error is 467123.4971024259
 r2 score 0.6856541750228613
----------------
using Ridge
mean squared error is 218200143734.61145
root mean squared error is 467118.9824173403
 r2 score 0.6856602512128132
----------------
using Lasso
mean squared error is 218203579422.18384
root mean squared error is 467122.6599322536
 r2 score 0.685655301751507
----------------
using DecisionTreeRegressor
mean squared error is 287527840268.7678
root mean squared error is 536216.2252942071
 r2 score 0.5857865740485747
----------------
using RandomForestRegressor
mean squared error is 43601073823.39975
root mean squared error is 208808.70150307374
 r2 score 0.9371881688163844
----------------
using KNeighborsRegressor
mean squared error is 66409314914.04476
root mean squared error is 257700.04833923638
 r2 score 0.9043305517131117
----------------
using SVR
mean squared error is 740439878298.0302
root mean squared error is 86048

## Hyperparameter Tuning

In [ ]:
rf={
    "n_estimators":[10,20,50,100,500],
    "criterion":["squared_error", "absolute_error", "friedman_mse", "poisson"],
    "max_depth":[5,10,7,8,13],
    "max_features" : ["sqrt", "log2", None]
}
knn={
    "n_neighbors":[4,5,7,10],
    "weights" : ['uniform', 'distance'],
    "algorithm" : ['auto', 'ball_tree', 'kd_tree', 'brute']
}

random_cv_model=[
    ("RandomForestRegressor",RandomForestRegressor(),rf),
    ("KNeighborsRegressor",KNeighborsRegressor(),knn)
]

In [32]:
from sklearn.model_selection import RandomizedSearchCV

for name,model,param in random_cv_model:
    rand=RandomizedSearchCV(estimator=model,param_distributions=param,n_jobs=-1,cv=5,verbose=2,n_iter=32)
    rand.fit(X_train,y_train)
    print(rand.best_params_)
    y_pred=rand.predict(X_test)
    print("using",name)
    model_performance(y_test,y_pred)

Fitting 5 folds for each of 32 candidates, totalling 160 fits


KeyboardInterrupt: 

In [ ]:
Fitting 5 folds for each of 32 candidates, totalling 160 fits
{'weights': 'distance', 'n_neighbors': 7, 'algorithm': 'auto'}
using KNeighborsRegressor
mean squared error is 55891761868.02064
root mean squared error is 236414.38591596036
 r2 score 0.91948216859914